In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


# ============================================================
# 1. Load the dataset
# ============================================================
df = pd.read_csv("/content/train.csv")

# Optional initial inspection
print("First 5 rows:")
print(df.head())
print("\nMissing values in original dataframe:")
print(df.isnull().sum())
print("\nOriginal dataframe info:")
df.info()


# ============================================================
# 2. Create a working copy for cleaning and feature engineering
# ============================================================
df_clean = df.copy()

# Drop columns that are likely not useful for a first baseline model
# - Cabin: too many missing values
# - Ticket: mostly identifier-like / hard to use directly
# - Name: text column, not used in this simple baseline
df_clean = df_clean.drop(columns=["Cabin", "Ticket", "Name"])

# Create a new feature: total family members traveling together
df_clean["family_size"] = df_clean["SibSp"] + df_clean["Parch"] + 1

# Fill missing Age values
# Earlier, I tried other things while debugging.
# Final choice here: fill missing Age with the mean.
df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].mean())
age_mean = df_clean["Age"].mean()
fare_mean = df_clean["Fare"].mean()

# Fill missing Embarked values with the most frequent category
df_clean["Embarked"] = df_clean["Embarked"].fillna(df_clean["Embarked"].mode()[0])
embarked_mode = df_clean["Embarked"].mode()[0]
# Encode Sex as binary
df_clean["Sex"] = df_clean["Sex"].map({"male": 0, "female": 1})

# ------------------------------------------------------------
# Note about Embarked:
# At first I tried mapping Embarked manually:
# df_clean["Embarked"] = df_clean["Embarked"].map({"S": 0, "C": 1, "Q": 2})
#
# But one-hot encoding is a better choice for categorical columns
# because it avoids introducing fake order.
# So I do NOT map Embarked manually in the final version.
# ------------------------------------------------------------

# One-hot encode categorical features
# Pclass is numeric-looking, but it is actually categorical
df_clean = pd.get_dummies(df_clean, columns=["Pclass", "Embarked"], drop_first=True)

print("\nCleaned dataframe info:")
df_clean.info()

print("\nMissing values after preprocessing:")
print(df_clean.isna().sum())


# ============================================================
# 3. Define a reusable evaluation function
# ============================================================
def evaluate_logistic_regression(X, y, experiment_name):
    """
    Train/test split, scale features, fit Logistic Regression,
    and print evaluation metrics.
    """
    print("\n" + "=" * 60)
    print(f"Experiment: {experiment_name}")
    print("=" * 60)
    #this print ============================================================

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Scale features
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Train model
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_scaled, y_train)

    # Predict on test set
    y_pred = model.predict(X_test_scaled)

    # Show a few predictions
    print("First 5 predictions vs actual values:")
    print("Predicted:", y_pred[:5])
    print("Actual   :", y_test.values[:5])

    # Evaluate
    print("\nAccuracy:")
    print(accuracy_score(y_test, y_pred))

    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification report:")
    print(classification_report(y_test, y_pred))


# ============================================================
# 4. Prepare target and run experiments
# ============================================================
y = df_clean["Survived"]

# ------------------------------------------------------------
# Experiment 1: baseline model
# Keep all currently available features except target
# ------------------------------------------------------------
X_baseline = df_clean.drop(columns=["Survived"])
evaluate_logistic_regression(X_baseline, y, "Baseline: all features except Survived")

# ------------------------------------------------------------
# Experiment 2: remove PassengerId
# PassengerId is just an identifier and should not help prediction
# ------------------------------------------------------------
X_no_passenger_id = X_baseline.drop(columns=["PassengerId"])
evaluate_logistic_regression(X_no_passenger_id, y, "Drop PassengerId")

# ------------------------------------------------------------
# Experiment 3: remove SibSp and Parch
# Since family_size was created, maybe these two become redundant
# ------------------------------------------------------------
X_family_only = X_no_passenger_id.drop(columns=["SibSp", "Parch"])
evaluate_logistic_regression(X_family_only, y, "Drop PassengerId, SibSp, and Parch")

test_set=pd.read_csv('/content/test.csv')
print(test_set.shape)
test_set.info()


def test_on_unseen_data(train_x, train_y, test_df):
    # Preprocess the test set in the same way as the training set

    test_df_clean = test_df.copy()
    passenger_ids = test_df_clean["PassengerId"]  # Save PassengerId for later if needed
    test_df_clean = test_df_clean.drop(columns=["Cabin", "Ticket", "Name","PassengerId"])
    test_df_clean["family_size"] = test_df_clean["SibSp"] + test_df_clean["Parch"] + 1

    test_df_clean["Age"] = test_df_clean["Age"].fillna(age_mean)

    test_df_clean["Embarked"] = test_df_clean["Embarked"].fillna(embarked_mode)

    test_df_clean= pd.get_dummies(test_df_clean, columns=["Pclass", "Embarked"], drop_first=True)
    test_df_clean= test_df_clean.drop(columns=["SibSp", "Parch"])
    test_df_clean["Sex"]=test_df_clean['Sex'].map({"male": 0, "female": 1})
    test_x=test_df_clean
    print(train_x.columns.tolist(), test_x.columns.tolist())
    print(train_x.columns.equals(test_x.columns))
    if not train_x.columns.equals(test_x.columns):
        test_x=test_x[train_x.columns]
    test_x["Fare"] = test_x["Fare"].fillna(fare_mean)
    print(test_x.isna().sum())

    scaler= StandardScaler()
    scaler.fit(train_x)
    train_x_scaled= scaler.transform(train_x)
    test_x_scaled= scaler.transform(test_x)
    model= LogisticRegression(max_iter=1000)
    model.fit(train_x_scaled, train_y)
    test_predictions= model.predict(test_x_scaled)
    submission =pd.DataFrame({
        "PassengerId": passenger_ids,
        "Survived": test_predictions
    })
    print("Test set predictions:")
    print(test_predictions[:5])
    submission .to_csv('submission.csv', index=False)


test_on_unseen_data(X_family_only, y, test_set)

First 5 rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN